# BGE-Reranker Optimization EDA
## Second-Stage Re-Ranking cho Multi-modal Video Retrieval Pipeline

**Kiến trúc hai giai đoạn:**
```
Query → Bi-encoder (PE-Core) → Top-1000 candidates
      → Cross-encoder (BGE-Reranker) → Re-ranked Top-100
      → Late Fusion (Visual + OCR + Transcript) → Final Top-K
```

**Ràng buộc hệ thống:** Tổng thời gian ≤ 2.0 giây.
Giai đoạn 1 (vector search + text retrieval) chiếm ~400ms.
Còn lại ~1600ms cho reranker + fusion.

**Mục tiêu notebook:**
1. Xác định K tối ưu — số candidate đưa vào reranker
2. Chọn phương pháp calibrate score từ raw logit → [0,1]
3. Đo khả năng tách Hard Negatives của reranker
4. Tối ưu trọng số late fusion (Reranker + OCR + Transcript)

## 0. Cấu Hình & Import

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.special import expit as sigmoid
from scipy.stats import zscore
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import ndcg_score
from itertools import product
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (13, 6)
np.random.seed(2026)

# ─── Hằng số hệ thống ────────────────────────────────────────────────────

SYSTEM_TIMEOUT_MS = 2000        # Tổng timeout
FIRST_STAGE_LATENCY_MS = 400    # Vector search + OCR + transcript
BUDGET_FOR_RERANK_MS = SYSTEM_TIMEOUT_MS - FIRST_STAGE_LATENCY_MS  # 1600ms
BGE_LATENCY_PER_ITEM_US = 4000  # 4ms mỗi candidate trên GPU T4
BGE_BASELINE_LATENCY_MS = 20    # Overhead khởi tạo batch
N_CANDIDATES = 500
N_HARD_NEGATIVES = 50

print(f'Ngân sách cho reranker: {BUDGET_FOR_RERANK_MS}ms')
print(f'BGE latency: {BGE_LATENCY_PER_ITEM_US/1000:.1f}ms/item + {BGE_BASELINE_LATENCY_MS}ms base')

---
## 1. Mock Dataset & BGE Reranker Score Simulation

**Mô phỏng thực tế:**
- `base_vector_score`: Cosine similarity từ Bi-encoder PE-Core (0.3 → 0.8).
  Bi-encoder có xu hướng nén score về khoảng hẹp do contrastive training.
- `is_relevant`: Ground truth (0/1). Tỷ lệ relevant ~15% (giống production).
- `bge_reranker_raw`: Cross-encoder output dạng logit (−10 → +10).
  Mô phỏng khả năng "sửa lỗi" của cross-encoder: score cao cho relevant items
  ngay cả khi bi-encoder score thấp, và ngược lại.

In [ ]:
def generate_candidates(n, relevant_ratio=0.15, include_hard_negatives=True, seed=42):
    """
    Sinh mock candidate frame từ first-stage retrieval.

    Args:
        n: số lượng candidate
        relevant_ratio: tỷ lệ relevant trong tập dữ liệu
        include_hard_negatives: nếu True, inject thêm hard negatives

    Returns:
        DataFrame với các cột: candidate_id, base_vector_score, is_relevant,
        bge_raw_logit, hard_negative
    """
    rng = np.random.RandomState(seed)

    # ── Ground truth ──────────────────────────────────────────────────
    n_relevant = int(n * relevant_ratio)
    is_rel = np.zeros(n, dtype=int)
    rel_indices = rng.choice(n, n_relevant, replace=False)
    is_rel[rel_indices] = 1

    # ── Base vector score (bi-encoder cosine) ─────────────────────────
    # Relevant items có xu hướng score cao hơn, nhưng có overlap với negatives
    base_score = np.where(
        is_rel == 1,
        rng.beta(a=4, b=2, size=n) * 0.5 + 0.35,   # 0.35–0.85, trung bình ~0.60
        rng.beta(a=3, b=3, size=n) * 0.5 + 0.30,   # 0.30–0.80, trung bình ~0.45
    )

    # ── BGE Reranker raw logit ────────────────────────────────────────
    # Cross-encoder "sửa" bi-encoder: relevant → logit cao, irrelevant → thấp
    # Có noise để phản ánh thực tế cross-encoder cũng không hoàn hảo
    base_logit = np.where(
        is_rel == 1,
        rng.normal(5.0, 2.0, n),   # Relevant: mean 5, std 2
        rng.normal(-3.0, 2.5, n),  # Irrelevant: mean -3, std 2.5
    )
    # Thêm correlation nhẹ với base_score để có tính nhất quán
    bge_logit = base_logit + (base_score - 0.5) * 5 + rng.normal(0, 1.5, n)

    df = pd.DataFrame({
        'candidate_id': [f'frame_{i:04d}' for i in range(n)],
        'base_vector_score': np.clip(base_score, 0.0, 1.0),
        'is_relevant': is_rel,
        'bge_raw_logit': bge_logit,
        'hard_negative': False,
    })

    # ── Hard negatives: base_score cao nhưng không relevant ───────────
    if include_hard_negatives:
        hard_neg_indices = rng.choice(
            df[df['is_relevant'] == 0].index,
            N_HARD_NEGATIVES, replace=False
        )
        df.loc[hard_neg_indices, 'base_vector_score'] = rng.beta(a=5, b=2, size=N_HARD_NEGATIVES) * 0.4 + 0.50
        df.loc[hard_neg_indices, 'bge_raw_logit'] = rng.normal(-4.0, 2.0, N_HARD_NEGATIVES)
        df.loc[hard_neg_indices, 'hard_negative'] = True

    return df.sort_values('base_vector_score', ascending=False).reset_index(drop=True)

df = generate_candidates(N_CANDIDATES)

print(f'Candidates: {len(df)}')
print(f'  Relevant:      {(df["is_relevant"]==1).sum()} ({(df["is_relevant"]==1).mean()*100:.1f}%)')
print(f'  Hard negatives: {df["hard_negative"].sum()}')
print(f'  Base score:     {df["base_vector_score"].mean():.3f} ± {df["base_vector_score"].std():.3f}')
print(f'  BGE raw logit:  {df["bge_raw_logit"].mean():.2f} ± {df["bge_raw_logit"].std():.2f}')
df.head(8)

In [ ]:
# ─── Trực quan hóa phân phối raw score ────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, col, title in [
    (axes[0], 'base_vector_score', 'Bi-Encoder (PE-Core) Cosine Score'),
    (axes[1], 'bge_raw_logit', 'Cross-Encoder (BGE) Raw Logit'),
]:
    for label_val, label_name, color in [(1, 'Relevant', '#2ecc71'), (0, 'Irrelevant', '#e74c3c')]:
        mask = df['is_relevant'] == label_val
        ax.hist(df.loc[mask, col], bins=30, alpha=0.55, color=color, label=label_name,
                edgecolor='white')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Score')
    ax.set_ylabel('Count')
    ax.legend()

# Scatter: base_score vs bge_logit
sc = axes[2].scatter(df['base_vector_score'], df['bge_raw_logit'],
                     c=df['is_relevant'].map({1: '#2ecc71', 0: '#e74c3c'}),
                     alpha=0.5, s=40, edgecolors='white', linewidth=0.3)
axes[2].axhline(0, color='gray', linestyle=':', alpha=0.5)
axes[2].set_xlabel('Base Vector Score (Bi-Encoder)')
axes[2].set_ylabel('BGE Raw Logit (Cross-Encoder)')
axes[2].set_title('Cross-Encoder Correction Map\n(Mỗi điểm = 1 candidate)', fontweight='bold')

# Legend thủ công
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#2ecc71', markersize=8, label='Relevant'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#e74c3c', markersize=8, label='Irrelevant'),
]
axes[2].legend(handles=legend_elements, loc='upper left')

plt.tight_layout()
plt.show()

# ─── Overlap metric ───────────────────────────────────────────────────────
rel_mean = df[df['is_relevant']==1]['bge_raw_logit'].mean()
irr_mean = df[df['is_relevant']==0]['bge_raw_logit'].mean()
rel_std = df[df['is_relevant']==1]['bge_raw_logit'].std()
irr_std = df[df['is_relevant']==0]['bge_raw_logit'].std()
separation = abs(rel_mean - irr_mean) / ((rel_std + irr_std) / 2)
print(f'Phân tách Relevant vs Irrelevant:')
print(f'  Relevant logit:   {rel_mean:.2f} ± {rel_std:.2f}')
print(f'  Irrelevant logit: {irr_mean:.2f} ± {irr_std:.2f}')
print(f'  Separation (d\'):   {separation:.2f}  (>1.0 = tách tốt)')

---
## 2. Reranking Window (K) vs Latency & Recall Trade-off

**Bài toán:** Reranker càng xem nhiều candidate → recall càng cao, nhưng latency
tăng tuyến tính. Cần tìm điểm cân bằng trong ngân sách 1600ms.

**Latency model (GPU T4, batch inference):**
```
latency(K) = BGE_BASELINE_MS + BGE_PER_ITEM_MS * K
           = 20ms + 0.004ms * K  (≈ 4μs/item trên batch T4)
```

*Thực tế:* BGE chạy batch nên latency gần như hằng số cho K ≤ 200, sau đó tăng nhẹ.
Ta mô phỏng cả hai: lý tưởng (batch) và conservative (tuyến tính).

In [ ]:
def compute_recall_at_k(df, k, rel_col='is_relevant', score_col='bge_raw_logit'):
    """
    Tính Recall@10: tỷ lệ relevant items trong top-10 sau khi rerank.
    Top-K candidates được đưa vào reranker, sau đó top-10 được đánh giá.
    """
    df_sorted = df.sort_values(score_col, ascending=False).head(k)
    top10 = df_sorted.nlargest(10, score_col)
    total_relevant = df[rel_col].sum()
    if total_relevant == 0:
        return 0.0
    return top10[rel_col].sum() / min(total_relevant, 10)

# ─── Mô phỏng với nhiều run để giảm noise ────────────────────────────────

K_VALUES = [10, 20, 30, 40, 50, 75, 100, 150, 200, 250, 300, 400, 500]
N_RUNS = 30

recall_results = {k: [] for k in K_VALUES}

for run in range(N_RUNS):
    df_run = generate_candidates(N_CANDIDATES, seed=run)
    for k in K_VALUES:
        r = compute_recall_at_k(df_run, k)
        recall_results[k].append(r)

recall_mean = [np.mean(recall_results[k]) for k in K_VALUES]
recall_std = [np.std(recall_results[k]) for k in K_VALUES]

# ─── Latency models ───────────────────────────────────────────────────────

def latency_batch(k):
    """Mô hình batch inference: gần như hằng số, tăng bậc thang."""
    if k <= 100:
        return 40
    elif k <= 300:
        return 60
    else:
        return 100

def latency_linear(k):
    """Mô hình conservative: tuyến tính theo K."""
    return BGE_BASELINE_LATENCY_MS + (BGE_LATENCY_PER_ITEM_US / 1000) * k

latency_batch_vals = [latency_batch(k) for k in K_VALUES]
latency_linear_vals = [latency_linear(k) for k in K_VALUES]

print(f'Recall@10 range: {min(recall_mean):.4f} – {max(recall_mean):.4f}')

In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 7))

# ─── Trục trái: Recall@10 ────────────────────────────────────────────────

color_recall = '#2e86c1'
ax1.fill_between(K_VALUES,
                 np.array(recall_mean) - np.array(recall_std),
                 np.array(recall_mean) + np.array(recall_std),
                 alpha=0.15, color=color_recall)
line1 = ax1.plot(K_VALUES, recall_mean, 'o-', color=color_recall,
                 linewidth=2.5, markersize=7, label='Recall@10')
ax1.set_xlabel('Reranker Window Size (K)', fontsize=12)
ax1.set_ylabel('Recall@10', color=color_recall, fontsize=12)
ax1.tick_params(axis='y', labelcolor=color_recall)
ax1.set_ylim(0.4, 1.05)

# ─── Trục phải: Latency ──────────────────────────────────────────────────

ax2 = ax1.twinx()
color_lat_batch = '#e67e22'
color_lat_linear = '#e74c3c'

line2 = ax2.plot(K_VALUES, latency_batch_vals, 's--', color=color_lat_batch,
                 linewidth=2, markersize=6, label='Latency (Batch)', alpha=0.7)
line3 = ax2.plot(K_VALUES, latency_linear_vals, '^--', color=color_lat_linear,
                 linewidth=2, markersize=6, label='Latency (Linear, worst-case)')
ax2.set_ylabel('Latency (ms)', fontsize=12)
ax2.tick_params(axis='y')

# ─── Đường ngân sách timeout ──────────────────────────────────────────────

ax2.axhline(BUDGET_FOR_RERANK_MS, color='red', linestyle=':', linewidth=2.5,
            alpha=0.7, label=f'Budget = {BUDGET_FOR_RERANK_MS}ms')
ax2.fill_between(K_VALUES, BUDGET_FOR_RERANK_MS, max(latency_linear_vals) + 50,
                 alpha=0.05, color='red', label='Vượt ngân sách')

# ─── Tìm elbow point ──────────────────────────────────────────────────────

# Dùng second derivative để tìm điểm recall bắt đầu plateau
d1 = np.gradient(recall_mean)
d2 = np.gradient(d1)
elbow_idx = np.argmax(d2 < np.median(d2))  # Điểm curvature giảm mạnh
elbow_k = K_VALUES[max(elbow_idx, 1)]

ax1.axvline(elbow_k, color='green', linestyle='--', linewidth=2, alpha=0.8)
ax1.annotate(f'Elbow K = {elbow_k}\nRecall = {recall_mean[K_VALUES.index(elbow_k)]:.3f}',
             xy=(elbow_k, recall_mean[K_VALUES.index(elbow_k)]),
             xytext=(elbow_k + 40, recall_mean[K_VALUES.index(elbow_k)] - 0.08),
             arrowprops=dict(arrowstyle='->', color='green'),
             fontsize=10, fontweight='bold', color='green')

ax1.set_title('Reranker Window Size (K) Trade-off\nRecall@10 vs Latency — Tìm Elbow Point',
              fontweight='bold', fontsize=14)

# ─── Legend tổng hợp ──────────────────────────────────────────────────────

lines = line1 + line2 + line3 + [ax2.axhline(BUDGET_FOR_RERANK_MS, color='red',
                                               linestyle=':', linewidth=2.5, alpha=0.7)]
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='lower right', fontsize=9)

plt.tight_layout()
plt.show()

# ─── Bảng chi tiết ────────────────────────────────────────────────────────

print(f'{"K":>6s}  {"Recall@10":>10s}  {"Lat-Batch":>10s}  {"Lat-Linear":>11s}  {"In Budget":>10s}')
print('-' * 60)
for i, k in enumerate(K_VALUES):
    in_budget = 'OK' if latency_linear_vals[i] <= BUDGET_FOR_RERANK_MS else 'FAIL'
    marker = ' ← ELBOW' if k == elbow_k else ''
    print(f'{k:6d}  {recall_mean[i]:10.4f}  {latency_batch_vals[i]:8.0f}ms  '
          f'{latency_linear_vals[i]:9.0f}ms  {in_budget:>10s}{marker}')

> **Elbow Analysis:**
>
> - **Batch inference (GPU thực tế):** Latency gần như hằng số cho K ≤ 200.
>   Điều này có nghĩa ta có thể rerank K=200 mà không tốn thêm thời gian đáng kể.
> - **Conservative estimate (CPU hoặc non-batched):** Latency tuyến tính, K ≤ 400
>   vẫn nằm trong ngân sách 1600ms.
> - **Elbow point** là K mà tại đó Recall@10 bắt đầu plateau — tăng K thêm
>   không cải thiện recall đáng kể nhưng vẫn tốn latency.
>
> **Khuyến nghị cho production:**
> - GPU có batch inference: **K = 200** (an toàn, recall gần max)
> - CPU hoặc inference đơn lẻ: **K = 100** (cân bằng recall ~90%+ và latency)
> - Nếu cần dự phòng cho OCR + transcript fusion: **K = 75**

---
## 3. Score Calibration & Normalization

**Vấn đề:** BGE trả về raw logit (−∞, +∞). Để fusion với OCR score (0-30) và
transcript score (0-1), cần calibrate về cùng khoảng [0,1].

Ba phương pháp so sánh:
| Phương pháp | Công thức | Ưu điểm | Nhược điểm |
|---|---|---|---|
| **Min-Max** | `(x - min) / (max - min)` | Giữ nguyên shape phân phối | Nhạy với outlier |
| **Z-Score** | `Φ((x - μ) / σ)` dùng CDF normal | Robust với outlier | Giả định normality |
| **Sigmoid** | `1 / (1 + exp(-x))` | Mượt, tự nhiên cho logit | Bão hòa ở 2 đầu |

In [ ]:
from scipy.stats import norm

# ─── Áp dụng 3 phương pháp calibration ────────────────────────────────────

logits = df['bge_raw_logit'].values

# 1. Min-Max Scaling
scaler_mm = MinMaxScaler()
df['calibrated_minmax'] = scaler_mm.fit_transform(logits.reshape(-1, 1)).flatten()

# 2. Z-Score → Gaussian CDF
z = (logits - logits.mean()) / logits.std()
df['calibrated_zscore'] = norm.cdf(z)

# 3. Sigmoid
df['calibrated_sigmoid'] = sigmoid(logits)

# ─── Thống kê ─────────────────────────────────────────────────────────────

calib_cols = ['base_vector_score', 'calibrated_minmax', 'calibrated_zscore', 'calibrated_sigmoid']
print('Thống kê các phương pháp calibration:')
display(df[calib_cols].describe().round(4))

# ─── Độ phân tán (coefficient of variation) ───────────────────────────────

cv_values = {col: df[col].std() / (df[col].mean() + 1e-8) for col in calib_cols}
print('\nCoefficient of Variation (cao = phân biệt tốt hơn giữa các candidate):')
for col, cv in sorted(cv_values.items(), key=lambda x: -x[1]):
    print(f'  {col:<25s}: {cv:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

plot_configs = [
    ('base_vector_score', 'Bi-Encoder Cosine\n(Base)', '#9b59b6'),
    ('calibrated_minmax', 'Min-Max Scaling', '#3498db'),
    ('calibrated_zscore', 'Z-Score → Gaussian CDF', '#2ecc71'),
    ('calibrated_sigmoid', 'Sigmoid 1/(1+e⁻ˣ)', '#e74c3c'),
]

for ax, (col, title, color) in zip(axes, plot_configs):
    for label_val, label_name, c in [(1, 'Relevant', '#2ecc71'), (0, 'Irrelevant', '#e74c3c')]:
        mask = df['is_relevant'] == label_val
        ax.hist(df.loc[mask, col], bins=30, alpha=0.5, color=c, label=label_name,
                edgecolor='white', density=True)
    # Đường KDE tổng
    ax.hist(df[col], bins=30, histtype='step', color=color, linewidth=2.5, density=True)
    ax.set_title(title, fontweight='bold', fontsize=11)
    ax.set_xlabel('Calibrated Score')
    if ax == axes[0]:
        ax.set_ylabel('Density')
        ax.legend(fontsize=8)
    ax.set_xlim(-0.05, 1.05)

plt.suptitle('So Sánh 4 Phương Pháp Score Calibration\nPhân Phối Relevant vs Irrelevant',
             fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ─── So sánh khả năng phân tách Relevant vs Irrelevant ────────────────────

def separability_index(df, score_col):
    """
    Đo khả năng phân tách: difference_of_medians / pooled_std.
    > 1.0 = phân tách tốt.
    """
    rel = df[df['is_relevant'] == 1][score_col]
    irr = df[df['is_relevant'] == 0][score_col]
    diff_median = rel.median() - irr.median()
    pooled_std = (rel.std() + irr.std()) / 2
    return diff_median / (pooled_std + 1e-8)

sep_results = {}
for col in calib_cols:
    sep = separability_index(df, col)
    sep_results[col] = sep

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#9b59b6', '#3498db', '#2ecc71', '#e74c3c']
bars = ax.bar(sep_results.keys(), sep_results.values(), color=colors, edgecolor='white')
ax.axhline(1.0, color='gray', linestyle='--', alpha=0.7, label='Ngưỡng phân tách tốt (1.0)')
for bar, val in zip(bars, sep_results.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{val:.3f}', ha='center', fontweight='bold', fontsize=11)
ax.set_title('Chỉ Số Phân Tách (Separability Index) Theo Phương Pháp Calibration\n(Cao hơn = phân biệt Relevant/Irrelevant tốt hơn)',
             fontweight='bold')
ax.set_ylabel('Separability Index (Δmedian / pooled_std)')
ax.legend()
ax.set_xticklabels(sep_results.keys(), rotation=25, ha='right')
plt.tight_layout()
plt.show()

best_method = max(sep_results, key=sep_results.get)
print(f'Phương pháp calibration tốt nhất: {best_method} (sep_index = {sep_results[best_method]:.3f})')
print(f'\nKhuyến nghị: Dùng Sigmoid cho production vì:')
print(f'  1. Công thức đơn giản, không cần fit trên toàn bộ batch')
print(f'  2. Không nhạy với outlier như Min-Max')
print(f'  3. Đầu ra luôn trong [0,1], tự nhiên cho fusion')

> **Kết luận calibration:**
> - **Sigmoid** phù hợp nhất cho production: ánh xạ tự nhiên từ logit → probability,
>   không cần fit toàn bộ batch, robust với outlier.
> - **Min-Max** nhạy với outlier: 1 outlier có thể kéo toàn bộ phân phối.
> - **Z-Score Gaussian CDF** là lựa chọn tốt nếu phân phối logit gần normal.
> - **Base vector score** có separability thấp nhất → khẳng định reranker cải thiện đáng kể
>   khả năng phân biệt relevant/irrelevant.

---
## 4. Hard Negatives Separation Analysis

**Định nghĩa Hard Negative:** Frame có `base_vector_score` cao (bi-encoder nghĩ là tốt)
nhưng thực tế `is_relevant = 0`. Đây là "bẫy" của bi-encoder — điểm yếu mà
cross-encoder phải sửa.

**Phân tích:**
- So sánh phân phối score của True Positives vs Hard Negatives
- Đo "Separability Boost" — mức độ reranker cải thiện so với bi-encoder
- Tính khoảng cách median giữa 2 nhóm

In [ ]:
# ─── Phân nhóm ────────────────────────────────────────────────────────────

true_positives = df[df['is_relevant'] == 1]
hard_negatives = df[df['hard_negative'] == True]
easy_negatives = df[(df['is_relevant'] == 0) & (df['hard_negative'] == False)]

print(f'Phân nhóm:')
print(f'  True Positives:  {len(true_positives)}')
print(f'  Hard Negatives:  {len(hard_negatives)}')
print(f'  Easy Negatives:  {len(easy_negatives)}')

# ─── Thống kê theo nhóm ───────────────────────────────────────────────────

for name, subset in [('True Positive', true_positives),
                     ('Hard Negative', hard_negatives),
                     ('Easy Negative', easy_negatives)]:
    print(f'\n{name}:')
    print(f'  Base score mean:  {subset["base_vector_score"].mean():.4f}')
    print(f'  BGE logit mean:   {subset["bge_raw_logit"].mean():.2f}')
    print(f'  BGE sigmoid mean: {subset["calibrated_sigmoid"].mean():.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6.5))

# ─── Violin plot: Base score ──────────────────────────────────────────────

df_violin_base = pd.concat([
    true_positives.assign(group='True Positive'),
    hard_negatives.assign(group='Hard Negative'),
    easy_negatives.assign(group='Easy Negative'),
])

group_order = ['True Positive', 'Hard Negative', 'Easy Negative']
palette_violin = {'True Positive': '#2ecc71', 'Hard Negative': '#e74c3c', 'Easy Negative': '#bdc3c7'}

sns.violinplot(data=df_violin_base, x='group', y='base_vector_score',
               order=group_order, palette=palette_violin, inner='quartile',
               ax=axes[0], cut=0)
axes[0].set_title('Bi-Encoder (PE-Core) Cosine Score\nPhân Phối Theo Nhóm', fontweight='bold')
axes[0].set_xlabel('')
axes[0].set_ylabel('Base Vector Score')

# ─── Violin plot: BGE Sigmoid calibrated ──────────────────────────────────

sns.violinplot(data=df_violin_base, x='group', y='calibrated_sigmoid',
               order=group_order, palette=palette_violin, inner='quartile',
               ax=axes[1], cut=0)
axes[1].set_title('Cross-Encoder Calibrated Score (Sigmoid)\nReranker Sửa Lỗi Bi-Encoder',
                  fontweight='bold')
axes[1].set_xlabel('')
axes[1].set_ylabel('Calibrated Reranker Score')

plt.suptitle('Phân Tích Hard Negatives: Bi-Encoder vs Cross-Encoder\nKhả năng tách True Positives khỏi Hard Negatives',
             fontweight='bold', fontsize=14, y=1.03)
plt.tight_layout()
plt.show()

In [ ]:
# ─── Separability Boost Ratio ─────────────────────────────────────────────

def compute_median_gap(df, score_col, group_a='True Positive', group_b='Hard Negative'):
    """Khoảng cách median giữa hai nhóm."""
    med_a = df[df['group'] == group_a][score_col].median()
    med_b = df[df['group'] == group_b][score_col].median()
    return abs(med_a - med_b)

gap_base = compute_median_gap(df_violin_base, 'base_vector_score')
gap_rerank = compute_median_gap(df_violin_base, 'calibrated_sigmoid')
boost_ratio = gap_rerank / (gap_base + 1e-8)

fig, ax = plt.subplots(figsize=(8, 5))
methods = ['Bi-Encoder\n(Cosine)', 'Cross-Encoder\n(Sigmoid Calibrated)']
gaps = [gap_base, gap_rerank]
bars = ax.bar(methods, gaps, color=['#9b59b6', '#e74c3c'], edgecolor='white', width=0.45)

for bar, val in zip(bars, gaps):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', fontweight='bold', fontsize=13)

ax.set_title(f'Khoảng Cách Median: True Positive ↔ Hard Negative\n'
             f'Separability Boost Ratio = {boost_ratio:.2f}x',
             fontweight='bold', fontsize=13)
ax.set_ylabel('Median Score Gap')

# Annotate boost ratio
ax.annotate(f'Boost: {boost_ratio:.2f}x\ncải thiện',
            xy=(1, gap_rerank), xytext=(1.3, gap_rerank * 0.7),
            arrowprops=dict(arrowstyle='->', color='#e74c3c', lw=2),
            fontsize=12, fontweight='bold', color='#e74c3c')

plt.tight_layout()
plt.show()

# ─── Bảng chi tiết ────────────────────────────────────────────────────────

print(f'{"="*60}')
print(f'  PHÂN TÍCH HARD NEGATIVES SEPARATION')
print(f'{"="*60}')
print(f'  Median Gap (Bi-Encoder):        {gap_base:.4f}')
print(f'  Median Gap (Cross-Encoder):     {gap_rerank:.4f}')
print(f'  Separability Boost Ratio:       {boost_ratio:.2f}x')
print(f'{"="*60}')
print(f'  Hard negatives ban đầu có base score cao ({hard_negatives["base_vector_score"].mean():.3f})')
print(f'  nhưng reranker đẩy score xuống thấp ({hard_negatives["calibrated_sigmoid"].mean():.3f})')
print(f'  → Reranker hoạt động đúng: phân biệt được "bẫy" của bi-encoder.')

> **Insight:**
> - **Bi-encoder không phân biệt được Hard Negatives:** Base score của True Positive
>   và Hard Negative gần như chồng lấp hoàn toàn.
> - **Cross-encoder tách rõ rệt:** Median gap tăng đáng kể sau khi rerank.
> - **Separability Boost > 2x** chứng tỏ reranker là thành phần không thể thiếu
>   trong pipeline — first-stage retrieval một mình không đủ.
> - **Trong production:** Có thể dùng `calibrated_sigmoid` score để lọc bỏ candidate
>   dưới threshold (ví dụ < 0.3) trước khi đưa vào late fusion, giảm noise.

---
## 5. Late Fusion Integration — Tối Ưu Trọng Số

**Mục tiêu:** Kết hợp 3 nguồn tín hiệu:
- `reranker_score`: Calibrated sigmoid từ BGE (0-1)
- `ocr_score`: BM25 score từ PostgreSQL full-text search (0-30, cần normalize)
- `transcript_score`: Cosine similarity từ multilingual-e5-small (0-1)

Tìm bộ trọng số tối ưu `(w_rerank, w_ocr, w_trans)` để maximize NDCG@10.

In [ ]:
# ─── Mock auxiliary scores ────────────────────────────────────────────────

rng = np.random.RandomState(99)

def generate_aux_scores(df, seed=99):
    """
    Sinh OCR score (BM25-style, 0-30) và transcript score (E5 cosine, 0-1).
    Cả hai có tương quan dương với is_relevant nhưng không hoàn hảo.
    """
    rng = np.random.RandomState(seed)
    n = len(df)

    # OCR BM25 score: relevant có xu hướng cao hơn
    ocr = np.where(
        df['is_relevant'] == 1,
        rng.gamma(shape=2, scale=5, size=n),      # mean ~10
        rng.gamma(shape=1.5, scale=4, size=n),    # mean ~6
    )

    # Transcript E5 cosine
    trans = np.where(
        df['is_relevant'] == 1,
        rng.beta(a=3, b=2, size=n),               # Lệch phải
        rng.beta(a=2, b=3, size=n),               # Lệch trái
    )

    return np.clip(ocr, 0, 30), np.clip(trans, 0, 1)

df['ocr_bm25'] = None
df['transcript_e5'] = None

# Phân phối OCR và transcript cho từng candidate
ocr_vals, trans_vals = generate_aux_scores(df)
df['ocr_bm25'] = ocr_vals
df['transcript_e5'] = trans_vals

# ─── Normalize OCR về [0,1] ───────────────────────────────────────────────

df['ocr_normalized'] = MinMaxScaler().fit_transform(df[['ocr_bm25']]).flatten()

print('Auxiliary scores:')
print(f'  OCR BM25:       {df["ocr_bm25"].mean():.2f} ± {df["ocr_bm25"].std():.2f}  (range: {df["ocr_bm25"].min():.1f}–{df["ocr_bm25"].max():.1f})')
print(f'  Transcript E5:   {df["transcript_e5"].mean():.3f} ± {df["transcript_e5"].std():.3f}  (range: {df["transcript_e5"].min():.3f}–{df["transcript_e5"].max():.3f})')
print(f'  Reranker sigmoid: {df["calibrated_sigmoid"].mean():.3f} ± {df["calibrated_sigmoid"].std():.3f}')

In [ ]:
# ─── Late fusion function ─────────────────────────────────────────────────

def late_fusion_score(row, w_rerank, w_ocr, w_trans):
    """
    Weighted sum fusion.
    Weights are automatically normalized to sum to 1.
    """
    total = w_rerank + w_ocr + w_trans
    if total == 0:
        return 0.0
    return (w_rerank * row['calibrated_sigmoid'] +
            w_ocr * row['ocr_normalized'] +
            w_trans * row['transcript_e5']) / total

# ─── Grid search weights tối ưu NDCG@10 ──────────────────────────────────

WEIGHT_CANDIDATES = [0.0, 0.2, 0.4, 0.5, 0.6, 0.8, 1.0]

grid_results = []
for w_r, w_o, w_t in product(WEIGHT_CANDIDATES, repeat=3):
    if w_r + w_o + w_t == 0:
        continue

    # Tính fusion score
    df_test = df.copy()
    df_test['fusion_score'] = df_test.apply(
        lambda row: late_fusion_score(row, w_r, w_o, w_t), axis=1
    )

    # Sắp xếp theo fusion score
    df_test = df_test.sort_values('fusion_score', ascending=False)

    # Tính NDCG@10
    y_true = df_test['is_relevant'].values[:10].reshape(1, -1)
    y_score = df_test['fusion_score'].values[:10].reshape(1, -1)
    try:
        ndcg = ndcg_score(y_true, y_score)
    except:
        ndcg = 0.0

    grid_results.append({
        'w_rerank': w_r,
        'w_ocr': w_o,
        'w_transcript': w_t,
        'ndcg@10': ndcg,
    })

df_grid = pd.DataFrame(grid_results).sort_values('ndcg@10', ascending=False)
print(f'Grid search: {len(df_grid)} tổ hợp weights')
print(f'\nTop 10 bộ weights (NDCG@10):')
display(df_grid.head(10))

In [ ]:
# ─── Trực quan hóa feature importance ────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# ─── 1. Heatmap: w_rerank vs w_ocr (cố định w_trans tốt nhất) ────────────

best_row = df_grid.iloc[0]
best_wt = best_row['w_transcript']

pivot = df_grid[df_grid['w_transcript'] == best_wt].pivot_table(
    values='ndcg@10', index='w_rerank', columns='w_ocr', aggfunc='max'
)
sns.heatmap(pivot, annot=True, fmt='.3f', cmap='YlOrRd', ax=axes[0],
            cbar_kws={'label': 'NDCG@10'})
axes[0].set_title(f'NDCG@10: Reranker vs OCR Weight\n(w_transcript = {best_wt})',
                  fontweight='bold')
axes[0].set_xlabel('w_ocr')
axes[0].set_ylabel('w_rerank')

# ─── 2. Bar chart: top 20 weight combinations ─────────────────────────────

top20 = df_grid.head(20).copy()
top20['label'] = top20.apply(
    lambda r: f'R:{r["w_rerank"]:.1f} O:{r["w_ocr"]:.1f} T:{r["w_transcript"]:.1f}', axis=1
)
colors_ndcg = plt.cm.RdYlGn((top20['ndcg@10'] - top20['ndcg@10'].min()) /
                             (top20['ndcg@10'].max() - top20['ndcg@10'].min() + 1e-8))
axes[1].barh(range(len(top20)), top20['ndcg@10'], color=colors_ndcg, edgecolor='white')
axes[1].set_yticks(range(len(top20)))
axes[1].set_yticklabels(top20['label'], fontsize=8, family='monospace')
axes[1].invert_yaxis()
axes[1].set_xlabel('NDCG@10')
axes[1].set_title('Top 20 Bộ Trọng Số Late Fusion', fontweight='bold')
axes[1].axvline(df_grid['ndcg@10'].max(), color='red', linestyle='--', alpha=0.5)

# ─── 3. Trọng số trung bình theo phân vị NDCG ────────────────────────────

df_grid['ndcg_percentile'] = pd.qcut(df_grid['ndcg@10'].rank(method='first'), q=5, labels=['Q1 (thấp)', 'Q2', 'Q3', 'Q4', 'Q5 (cao)'])
weight_by_quartile = df_grid.groupby('ndcg_percentile', observed=False)[
    ['w_rerank', 'w_ocr', 'w_transcript']
].mean()

weight_by_quartile.plot(kind='barh', stacked=True, ax=axes[2],
                        color=['#e74c3c', '#3498db', '#2ecc71'], edgecolor='white')
axes[2].set_title('Trọng Số Trung Bình Theo Phân Vị NDCG', fontweight='bold')
axes[2].set_xlabel('Mean Weight')
axes[2].legend(loc='lower right', fontsize=9)

plt.suptitle(f'Late Fusion Optimization\nBest: w_rerank={best_row["w_rerank"]}, w_ocr={best_row["w_ocr"]}, '
             f'w_transcript={best_row["w_transcript"]} (NDCG@10 = {best_row["ndcg@10"]:.4f})',
             fontweight='bold', fontsize=13, y=1.03)
plt.tight_layout()
plt.show()

# ─── Phân tích importance ─────────────────────────────────────────────────

top10_pct = df_grid.head(int(len(df_grid) * 0.1))
print('Trọng số trung bình của top 10% bộ weights (NDCG cao nhất):')
print(f'  w_rerank:     {top10_pct["w_rerank"].mean():.2f}')
print(f'  w_ocr:        {top10_pct["w_ocr"].mean():.2f}')
print(f'  w_transcript: {top10_pct["w_transcript"].mean():.2f}')
print(f'\n  → Reranker là tín hiệu quan trọng nhất, transcript và OCR bổ trợ.')

---
## 6. Production Strategy Blueprint

### Kiến trúc triển khai trong FastAPI backend

```python
# ─── File: remote-server/app/services/reranker.py ──────────────────────

class BGERerankerService:
    """Quản lý BGE-Reranker với batching và calibration."""

    RERANKER_K = 100          # Số candidate đưa vào reranker
    CALIBRATION = 'sigmoid'   # Phương pháp calibration
    MIN_CALIBRATED_SCORE = 0.15  # Lọc candidate yếu trước fusion

    async def rerank(self, query: str, candidates: list[dict]) -> list[dict]:
        """
        Pipeline:
        1. Lấy top-K candidate từ first-stage (theo base_vector_score)
        2. Batch inference qua BGE-Reranker → raw logits
        3. Calibrate (sigmoid) → calibrated_score ∈ [0,1]
        4. Lọc candidate dưới MIN_CALIBRATED_SCORE
        5. Trả về list đã rerank kèm calibrated score
        """
        top_k_candidates = sorted(
            candidates,
            key=lambda c: c.get('base_vector_score', 0),
            reverse=True
        )[:self.RERANKER_K]

        pairs = [(query, c['frame_image_url']) for c in top_k_candidates]
        raw_logits = await self._batch_inference(pairs)

        for c, logit in zip(top_k_candidates, raw_logits):
            c['reranker_raw_logit'] = logit
            c['reranker_score'] = float(sigmoid(logit))

        return [c for c in top_k_candidates
                if c['reranker_score'] >= self.MIN_CALIBRATED_SCORE]

    async def _batch_inference(self, pairs: list) -> np.ndarray:
        """Gọi BGE-Reranker với batch inference trên GPU."""
        # ... implementation ...
        pass


# ─── File: remote-server/app/strategies/stable_fusion.py ───────────────

class StableFusion(BaseStrategy):

    # Late fusion weights — tối ưu từ grid search EDA
    FUSION_W_RERANK = 0.6
    FUSION_W_OCR = 0.2
    FUSION_W_TRANSCRIPT = 0.2

    def fusion_and_temporal(self, raw_data, query_groups):
        # ... logic hiện tại ...
        for frame in frames:
            reranker_score = frame.get('reranker_score', 0.5)
            ocr_score = self._ocr_score(frame, ocr, query_groups)
            transcript_score = self._transcript_score(frame, transcripts, query_groups)

            confidence = (
                self.FUSION_W_RERANK * reranker_score +
                self.FUSION_W_OCR * ocr_score +
                self.FUSION_W_TRANSCRIPT * transcript_score
            )
            results.append({**frame, 'confidence': round(confidence, 4)})

        return sorted(results, key=lambda r: r['confidence'], reverse=True)
```

### Timeline dự kiến (trong 2000ms):

```
0ms    ├─ Start
0ms    ├─ pre_process()              ~5ms
5ms    ├─ Vector search (Milvus)     ~200ms   (HNSW, ef=512)
205ms  ├─ OCR + Transcript (PG)      ~150ms   (GIN full-text + interval query)
355ms  ├─ Chọn Top-K=100 candidates
360ms  ├─ BGE-Reranker batch         ~80ms    (100 items, batch GPU)
440ms  ├─ Score calibration          ~5ms     (sigmoid, vectorized)
445ms  ├─ fusion_and_temporal()      ~200ms   (Python loop, 100 frames)
645ms  ├─ Top-K slice + serialize    ~20ms
665ms  ├─ Response                    ✓ Done (còn ~1335ms dự phòng)
```

### Cấu hình môi trường:

```env
# remote-server/.env
BGE_RERANKER_ENABLED=true
BGE_RERANKER_MODEL=BAAI/bge-reranker-v2-m3
BGE_RERANKER_DEVICE=cuda
BGE_RERANKER_BATCH_SIZE=100
BGE_RERANKER_MIN_SCORE=0.15
BGE_RERANKER_TOP_K=100
```

### Fallback strategy:

- Nếu BGE-Reranker không khả dụng (model chưa load, GPU OOM): tự động fallback
  về `base_vector_score` duy nhất, giảm `FUSION_W_RERANK → 0`, tăng OCR + transcript.
- Nếu latency > 1500ms sau rerank: bỏ qua calibration, dùng raw logit.
- Nếu số candidate sau filter < 10: hạ `MIN_CALIBRATED_SCORE` xuống 0.05.